# Stage 4G — Final Reproducible Full-Cohort Models

**Purpose:** fit and package the already frozen final model candidates using all 820 development patients. This stage creates deployable research artifacts; it does **not** estimate performance.

Scientific hierarchy:

- Primary: Stage 3B common-core ridge for all three outcomes.
- Secondary: broad ridge for all three outcomes.
- Exploratory challenger: XGBoost for `NoHandJointInvolvement` and `AsymmetricArthritis`.

All performance claims remain based on repeated nested out-of-fold estimates from Stages 3B–4. This notebook never computes or reports apparent full-cohort performance. `OnlyLargeJoints` remains excluded.

In [ ]:
# 1. Imports and deterministic settings
from __future__ import annotations
import hashlib,importlib,json,os,platform,shutil,subprocess,sys,time,warnings
from dataclasses import dataclass
from pathlib import Path
from packaging.version import Version
import joblib,numpy as np,pandas as pd,yaml
import matplotlib.pyplot as plt
from IPython.display import display,Markdown
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV,RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from threadpoolctl import threadpool_limits
import xgboost
from xgboost import XGBClassifier
if Version(sklearn.__version__)<Version('1.8') or Version(sklearn.__version__)>=Version('1.10'):raise RuntimeError(f'scikit-learn >=1.8,<1.10 required; got {sklearn.__version__}')
MASTER_SEED=20261020
np.random.seed(MASTER_SEED);pd.set_option('display.max_columns',180);pd.set_option('display.width',220)
print('Python',sys.version.split()[0],'| sklearn',sklearn.__version__,'| xgboost',xgboost.__version__)

In [ ]:
# 2. Project discovery, mode, and isolated outputs
NOTEBOOK_NAME='04G_final_model_packaging.ipynb';ENV_PROJECT = Path(os.environ['RA_PROJECT_DIR']).expanduser() if os.environ.get('RA_PROJECT_DIR') else Path.cwd();COLAB_PROJECT=Path.cwd()
env=os.environ.get('RA_PROJECT_DIR');candidates=([Path(env)] if env else [])+[ENV_PROJECT,COLAB_PROJECT,Path.cwd(),Path.cwd()/'ra_analysis',Path.cwd()/'stage2_outputs']
PROJECT_DIR=next((p for p in candidates if (p/'derived_data').exists() and (p/'modeling_results').exists()),None)
if PROJECT_DIR is None:raise FileNotFoundError('Set RA_PROJECT_DIR to the project containing derived_data and modeling_results.')
DERIVED=PROJECT_DIR/'derived_data';S3B=PROJECT_DIR/'modeling_results'/'tables';S3CROOT=PROJECT_DIR/'modeling_results_stage3c';S3C=next((p for p in [S3CROOT/'tables',S3CROOT/'tables-3c'] if p.exists()),None)
QUICK=os.environ.get('RA_STAGE4G_QUICK_TEST','0')=='1';FORCE=os.environ.get('RA_STAGE4G_FORCE_RECOMPUTE','0')=='1';XGB_DEVICE=os.environ.get('RA_STAGE4G_XGB_DEVICE','cpu')
OUT=PROJECT_DIR/('modeling_results_stage4g_quick' if QUICK else 'modeling_results_stage4g');PACKAGE=OUT/'final_model_package';TABLES=OUT/'tables';FIGURES=OUT/'figures';REPORT=OUT/'report'
for d in [OUT,PACKAGE,TABLES,FIGURES,REPORT]:d.mkdir(parents=True,exist_ok=True)
protocol_candidates=[PROJECT_DIR/'stage4g_final_model_protocol.yaml',PROJECT_DIR/'config'/'stage4g_final_model_protocol.yaml',Path.cwd()/'stage4g_final_model_protocol.yaml',ENV_PROJECT/'stage4g_final_model_protocol.yaml']
PROTOCOL_PATH=next((p for p in protocol_candidates if p.exists()),None)
component_candidates=[PROJECT_DIR/'ra_final_model_components.py',Path.cwd()/'ra_final_model_components.py',ENV_PROJECT/'ra_final_model_components.py']
COMPONENT_PATH=next((p for p in component_candidates if p.exists()),None)
if PROTOCOL_PATH is None or COMPONENT_PATH is None:raise FileNotFoundError('Stage 4G protocol or portable component module missing')
CV_REPEATS=2 if QUICK else 10;CV_SPLITS=3 if QUICK else 5;XGB_N=4 if QUICK else 50;N_JOBS=-1
print('Project:',PROJECT_DIR.resolve());print('Output:',OUT.resolve());print('Mode:','QUICK SOFTWARE TEST' if QUICK else 'FULL FINAL MODEL FIT')

In [ ]:
# 3. Import portable components from the exact distributed module
sys.path.insert(0,str(COMPONENT_PATH.parent))
from ra_final_model_components import MissingnessFlags,RADurationLogTransformer
shutil.copy2(COMPONENT_PATH,PACKAGE/'ra_final_model_components.py')
with open(PROTOCOL_PATH,encoding='utf-8') as f:PROTOCOL=yaml.safe_load(f)

In [ ]:
# 4. Prior audits, immutable inputs, and signed run definition
def first(paths,label):
 p=next((Path(x) for x in paths if Path(x).exists()),None)
 if p is None:raise FileNotFoundError(label)
 return p
def sha(path):
 h=hashlib.sha256();
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
 return h.hexdigest()
inputs={'stage3b_audit':first([S3B/'stage3b_validation_audit.csv',S3B/'stage3b__validation_audit.csv'],'Stage 3B audit'),
 'stage3b_performance':first([S3B/'point_estimate_performance.csv',S3B/'stage3b__point_estimate_performance.csv'],'Stage 3B performance'),
 'stage3c_audit':first([S3C/'stage3c_validation_audit.csv'],'Stage 3C audit'),
 'stage3c_performance':first([S3C/'point_estimate_performance.csv'],'Stage 3C performance')}
for p in [inputs['stage3b_audit'],inputs['stage3c_audit']]:
 a=pd.read_csv(p)
 if not a.status.eq('PASS').all():raise AssertionError(f'Prior audit failed: {p}')
for o in ['NegRFandACCP','NoHandJointInvolvement','AsymmetricArthritis']:
 inputs[f'{o}_data']=DERIVED/f'{o}__primary_eligible.csv';inputs[f'{o}_manifest']=DERIVED/f'{o}__predictor_manifest.csv'
 for p in [inputs[f'{o}_data'],inputs[f'{o}_manifest']]:
  if not p.exists():raise FileNotFoundError(p)
payload={'pipeline_version':'4G-1.0-final-package','mode':'quick' if QUICK else 'full','protocol_sha256':sha(PROTOCOL_PATH),'component_sha256':sha(COMPONENT_PATH),'inputs':{k:sha(v) for k,v in inputs.items()},'cv_repeats':CV_REPEATS,'cv_splits':CV_SPLITS,'xgb_candidates':XGB_N,'seed':MASTER_SEED,'sklearn':sklearn.__version__,'xgboost':xgboost.__version__}
RUN_SIGNATURE=hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()
with open(OUT/'run_signature.json','w') as f:json.dump({'run_signature':RUN_SIGNATURE,**payload},f,indent=2)
print('Run signature:',RUN_SIGNATURE)

## Frozen model definitions and portable raw-input pipeline

In [ ]:
# 5. Model registry
OUTCOMES=['NegRFandACCP','NoHandJointInvolvement','AsymmetricArthritis'];CORE_RAW=['Age','Gender','RADurationDays','DASJointCountTEN28','DASJointCountSW','PatientGlobalAssessment','AbnormalCRPorESR']
@dataclass(frozen=True)
class FinalSpec:
 outcome:str;model:str;status:str;scope:str
SPECS=[]
for o in OUTCOMES:
 SPECS += [FinalSpec(o,'core_ridge','primary','core'),FinalSpec(o,'refined_broad_ridge','secondary','broad')]
 if o!='NegRFandACCP':SPECS += [FinalSpec(o,'xgboost','exploratory_challenger','broad')]
assert len(SPECS)==8
for s in SPECS:print(s)

In [ ]:
# 6. Data and preprocessing helpers
CATEGORY_LEVELS={'Gender':['Female','Male'],'RAOnset':['Insidious','Acute','Unknown'],'PhysicalActivity':['Moderate','Low','High']}
def load_data(spec):
 data=pd.read_csv(inputs[f'{spec.outcome}_data']);manifest=pd.read_csv(inputs[f'{spec.outcome}_manifest'])
 raw=CORE_RAW.copy() if spec.scope=='core' else manifest.loc[manifest.role.eq('primary_predictor'),'column'].tolist()
 if spec.outcome in raw or 'VisitID' in raw or 'OnlyLargeJoints' in raw:raise AssertionError('Leakage/excluded outcome')
 semantic=dict(zip(manifest.column,manifest.semantic_type));semantic['RADurationDays_log1p']='numeric';semantic.pop('RADurationDays',None)
 transformed=['RADurationDays_log1p' if x=='RADurationDays' else x for x in raw]
 if len(data)!=820 or not data.VisitID.is_unique:raise AssertionError('Unexpected population')
 return data,raw,transformed,semantic,manifest

def make_preprocessor(engineered,transformed,semantic):
 numeric=[x for x in transformed if semantic[x]=='numeric'];binary=[x for x in transformed if semantic[x]=='binary'];categorical=[x for x in transformed if semantic[x]=='categorical'];indicators=[x for x in numeric+binary if engineered[x].isna().any()];tr=[]
 if numeric:tr.append(('numeric',Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('scaler',StandardScaler())]),numeric))
 if binary:tr.append(('binary',Pipeline([('imputer',SimpleImputer(strategy='most_frequent',keep_empty_features=True))]),binary))
 if categorical:
  categories=[]
  for col in categorical:
   if col not in CATEGORY_LEVELS:raise KeyError(col)
   observed=set(engineered[col].dropna().astype(str).unique());allowed=set(CATEGORY_LEVELS[col])-{'Unknown'}
   if not observed.issubset(allowed):raise ValueError((col,observed-allowed))
   categories.append(CATEGORY_LEVELS[col])
  tr.append(('categorical',Pipeline([('imputer',SimpleImputer(strategy='constant',fill_value='Unknown',keep_empty_features=True)),('onehot',OneHotEncoder(categories=categories,drop='first',handle_unknown='ignore',sparse_output=True))]),categorical))
 if indicators:tr.append(('missing',MissingnessFlags(),indicators))
 return ColumnTransformer(tr,remainder='drop',verbose_feature_names_out=True),{'numeric':numeric,'binary':binary,'categorical':categorical,'missing':indicators}

In [ ]:
# 7. Frozen tuning spaces
CORE_C=[0.01,1,100] if QUICK else np.logspace(-4,4,13).tolist()
BROAD_C=[0.01,0.1,0.5] if QUICK else np.logspace(np.log10(0.003),np.log10(0.5),21).tolist()
XGB_SPACE={'model__max_depth':[1,2] if QUICK else [1,2,3],'model__learning_rate':[0.05,0.1] if QUICK else [0.01,0.03,0.05,0.1],
 'model__min_child_weight':[5] if QUICK else [5,10,20],'model__subsample':[0.8] if QUICK else [0.7,0.85,1.0],
 'model__colsample_bytree':[0.75] if QUICK else [0.5,0.75,1.0],'model__reg_lambda':[10] if QUICK else [1,10,50],
 'model__reg_alpha':[0] if QUICK else [0,0.1,1],'model__n_estimators':[150] if QUICK else [300,600,1000]}
def estimator_and_space(spec):
 if spec.model in ['core_ridge','refined_broad_ridge']:
  return LogisticRegression(solver='lbfgs',max_iter=10000,random_state=MASTER_SEED),{'model__C':CORE_C if spec.model=='core_ridge' else BROAD_C},'grid'
 return XGBClassifier(objective='binary:logistic',eval_metric='logloss',tree_method='hist',device=XGB_DEVICE,random_state=MASTER_SEED,n_jobs=1,scale_pos_weight=1),XGB_SPACE,'random'

In [ ]:
# 8. One-standard-error selection rules
def tuning_frame(search):
 cv=search.cv_results_;split_cols=sorted([c for c in cv if c.startswith('split') and c.endswith('_test_score')],key=lambda x:int(x.split('_')[0][5:]))
 losses=-np.column_stack([cv[c] for c in split_cols]);out=pd.DataFrame({'candidate_index':np.arange(len(cv['params'])),'mean_cv_log_loss':losses.mean(1),'sd_cv_log_loss':losses.std(1,ddof=1),'se_cv_log_loss':losses.std(1,ddof=1)/np.sqrt(losses.shape[1]),'params_json':[json.dumps(p,sort_keys=True,default=float) for p in cv['params']]})
 out['within_one_se']=False;best=int(out.mean_cv_log_loss.idxmin());cut=float(out.loc[best,'mean_cv_log_loss']+out.loc[best,'se_cv_log_loss']);out.loc[out.mean_cv_log_loss.le(cut+1e-15),'within_one_se']=True;return out,best,cut

def choose_candidate(spec,t):
 eligible=t[t.within_one_se].copy()
 if spec.model in ['core_ridge','refined_broad_ridge']:
  eligible['C']=eligible.params_json.map(lambda s:json.loads(s)['model__C']);return int(eligible.sort_values(['C','candidate_index']).iloc[0].candidate_index)
 def key(r):
  p=json.loads(r.params_json);return (p['model__max_depth'],p['model__n_estimators'],-p['model__min_child_weight'],-p['model__reg_lambda'],-p['model__reg_alpha'],p['model__learning_rate'],-p['model__subsample'],-p['model__colsample_bytree'],r.candidate_index)
 return int(min((r for _,r in eligible.iterrows()),key=key).candidate_index)

## Fit, document, serialize, and reload every final model

In [ ]:
# 9. Documentation helpers
def model_id(s):return f'{s.outcome}__{s.model}__{s.status}'
def schema_rows(data,raw,manifest,meta):
 sem=dict(zip(manifest.column,manifest.semantic_type));rows=[]
 for col in raw:
  vals=CATEGORY_LEVELS.get(col,[]);rows.append({'predictor':col,'semantic_type':sem[col],'allowed_categories_json':json.dumps(vals),'training_missing_n':int(data[col].isna().sum()),'training_missing_fraction':float(data[col].isna().mean()),'imputation':'median' if sem[col]=='numeric' else ('most_frequent' if sem[col]=='binary' else 'Unknown category'),'transformation':'log1p then standardize' if col=='RADurationDays' else ('standardize' if sem[col]=='numeric' else ('fixed-level one-hot, first level reference' if sem[col]=='categorical' else 'none')),'explicit_missingness_indicator':('RADurationDays_log1p' if col=='RADurationDays' else col) in meta['missing']})
 return pd.DataFrame(rows)
def synthetic_example(data,raw,manifest):
 sem=dict(zip(manifest.column,manifest.semantic_type));z={}
 for col in raw:
  s=data[col]
  if sem[col]=='numeric':z[col]=float(s.median())
  elif sem[col]=='binary':z[col]=float(s.mode(dropna=True).iloc[0])
  else:z[col]=str(s.mode(dropna=True).iloc[0])
 # Exercise one permitted missing value without using a real patient.
 miss=[c for c in raw if data[c].isna().any()]
 if miss:z[miss[0]]=np.nan
 return pd.DataFrame([z])

In [ ]:
# 10. Fit/resume one signed final package
all_tuning=[];registry_rows=[];validation_rows=[]
for spec_index,spec in enumerate(SPECS):
 mid=model_id(spec);mdir=PACKAGE/mid;mdir.mkdir(parents=True,exist_ok=True);model_path=mdir/'pipeline.joblib';meta_path=mdir/'metadata.json'
 data,raw,transformed,semantic,manifest=load_data(spec);X=data[raw].copy();y=data[spec.outcome].astype(int).to_numpy();engineered=RADurationLogTransformer().fit_transform(X)
 prep,prep_meta=make_preprocessor(engineered,transformed,semantic);est,space,kind=estimator_and_space(spec);pipe=Pipeline([('feature_engineering',RADurationLogTransformer()),('preprocess',prep),('model',est)])
 can_load=False
 if model_path.exists() and meta_path.exists() and not FORCE:
  old=json.loads(meta_path.read_text(encoding='utf-8'));can_load=old.get('run_signature')==RUN_SIGNATURE
 if can_load:
  fitted=joblib.load(model_path);tune=pd.read_csv(mdir/'tuning_candidates.csv');selected=int(json.loads(meta_path.read_text())['selected_candidate_index']);print('Loaded',mid)
 else:
  cv=RepeatedStratifiedKFold(n_splits=CV_SPLITS,n_repeats=CV_REPEATS,random_state=MASTER_SEED+spec_index)
  common=dict(estimator=pipe,scoring='neg_log_loss',cv=cv,refit=False,n_jobs=N_JOBS,pre_dispatch='2*n_jobs',error_score='raise',return_train_score=False)
  search=GridSearchCV(param_grid=space,**common) if kind=='grid' else RandomizedSearchCV(param_distributions=space,n_iter=XGB_N,random_state=MASTER_SEED,**common)
  t0=time.time()
  with warnings.catch_warnings(record=True) as caught:
   warnings.simplefilter('always')
   with threadpool_limits(limits=1,user_api='blas'):search.fit(X,y)
  conv=[str(w.message) for w in caught if issubclass(w.category,ConvergenceWarning)]
  if conv:raise RuntimeError(f'Convergence warning in {mid}: {conv}')
  tune,best,cut=tuning_frame(search);selected=choose_candidate(spec,tune);params=json.loads(tune.loc[tune.candidate_index.eq(selected),'params_json'].iloc[0]);fitted=clone(pipe).set_params(**params).fit(X,y)
  tune['minimum_mean_candidate']=tune.candidate_index.eq(best);tune['selected_by_one_se_rule']=tune.candidate_index.eq(selected);tune['one_se_cutoff']=cut;tune['outcome']=spec.outcome;tune['model']=spec.model;tune['status']=spec.status;tune['fit_search_seconds']=time.time()-t0
  tune.to_csv(mdir/'tuning_candidates.csv',index=False,encoding='utf-8-sig');tmp=mdir/'pipeline.tmp.joblib';joblib.dump(fitted,tmp,compress=3);tmp.replace(model_path)
  print('Fitted',mid,f'({time.time()-t0:.1f}s)')
 if 'outcome' not in tune:tune['outcome']=spec.outcome;tune['model']=spec.model;tune['status']=spec.status
 all_tuning.append(tune)
 params=fitted.named_steps['model'].get_params();selected_params={k.replace('model__',''):v for k,v in json.loads(tune.loc[tune.candidate_index.eq(selected),'params_json'].iloc[0]).items()}
 feature_names=fitted.named_steps['preprocess'].get_feature_names_out().tolist();schema=schema_rows(data,raw,manifest,prep_meta);schema.to_csv(mdir/'predictor_dictionary.csv',index=False,encoding='utf-8-sig');pd.DataFrame({'raw_predictor':raw}).to_csv(mdir/'raw_predictors.csv',index=False,encoding='utf-8-sig');pd.DataFrame({'transformed_feature':feature_names}).to_csv(mdir/'transformed_features.csv',index=False,encoding='utf-8-sig')
 if spec.model!='xgboost':pd.DataFrame({'transformed_feature':feature_names,'coefficient':fitted.named_steps['model'].coef_.ravel()}).to_csv(mdir/'coefficients.csv',index=False,encoding='utf-8-sig')
 example=synthetic_example(data,raw,manifest);example.to_csv(mdir/'example_input.csv',index=False,encoding='utf-8-sig');prob=float(fitted.predict_proba(example)[:,1][0]);pd.DataFrame({'row_id':[0],'probability':[prob]}).to_csv(mdir/'expected_example_output.csv',index=False,encoding='utf-8-sig')
 metadata={'model_id':mid,'outcome':spec.outcome,'model':spec.model,'status':spec.status,'scope':spec.scope,'run_signature':RUN_SIGNATURE,'n_development':len(data),'events':int(y.sum()),'raw_predictors':raw,'transformed_feature_count':len(feature_names),'selected_candidate_index':selected,'selected_hyperparameters':selected_params,'cv_splits':CV_SPLITS,'cv_repeats':CV_REPEATS,'tuning_objective':'log_loss','selection_rule':'one-standard-error stability rule','intended_use':PROTOCOL['intended_use'],'performance_source':'Prior repeated nested OOF validation; no apparent performance calculated in Stage 4G','seed':MASTER_SEED,'software':{'python':sys.version.split()[0],'numpy':np.__version__,'pandas':pd.__version__,'scikit_learn':sklearn.__version__,'xgboost':xgboost.__version__,'joblib':joblib.__version__}}
 meta_path.write_text(json.dumps(metadata,indent=2,default=float),encoding='utf-8')
 reloaded=joblib.load(model_path);reloaded_prob=float(reloaded.predict_proba(example)[:,1][0]);validation_rows.append({'model_id':mid,'expected_probability':prob,'reloaded_probability':reloaded_prob,'absolute_difference':abs(prob-reloaded_prob),'reload_pass':abs(prob-reloaded_prob)<1e-12})
 registry_rows.append({'model_id':mid,'outcome':spec.outcome,'model':spec.model,'status':spec.status,'scope':spec.scope,'n':len(data),'events':int(y.sum()),'raw_predictor_count':len(raw),'transformed_feature_count':len(feature_names),'selected_hyperparameters_json':json.dumps(selected_params,sort_keys=True,default=float),'pipeline_path':str(model_path)})

In [ ]:
# 11. Consolidated tuning, model registry, and reload validation
all_tuning=pd.concat(all_tuning,ignore_index=True);model_registry=pd.DataFrame(registry_rows);reload_validation=pd.DataFrame(validation_rows)
all_tuning.to_csv(TABLES/'all_final_tuning_candidates.csv',index=False,encoding='utf-8-sig');model_registry.to_csv(TABLES/'final_model_registry.csv',index=False,encoding='utf-8-sig');reload_validation.to_csv(TABLES/'reload_prediction_validation.csv',index=False,encoding='utf-8-sig')
display(model_registry);display(reload_validation)

In [ ]:
# 12. Attach prior OOF evidence without calculating full-data performance
s3b_perf=pd.read_csv(inputs['stage3b_performance']);s3c_perf=pd.read_csv(inputs['stage3c_performance']);evidence=[]
for spec in SPECS:
 src=s3b_perf.query("outcome==@spec.outcome and model=='core_ridge'") if spec.model=='core_ridge' else s3c_perf.query('outcome==@spec.outcome and model==@spec.model')
 if len(src)!=1:raise AssertionError((spec,src.shape))
 r=src.iloc[0];row={'model_id':model_id(spec),'outcome':spec.outcome,'model':spec.model,'status':spec.status,'evidence_type':'previous repeated nested OOF internal validation'}
 for k in ['auroc','average_precision','log_loss','brier','calibration_intercept','calibration_slope']:
  if k in r:row[k]=r[k]
 evidence.append(row)
evidence=pd.DataFrame(evidence);evidence.to_csv(TABLES/'prior_oof_performance_reference.csv',index=False,encoding='utf-8-sig');display(evidence)

In [ ]:
# 13. Generate model cards
for _,r in model_registry.iterrows():
 mdir=PACKAGE/r.model_id;meta=json.loads((mdir/'metadata.json').read_text());ev=evidence[evidence.model_id.eq(r.model_id)].iloc[0]
 card=f"""# Model Card — {r.model_id}

## Status

- Outcome: `{r.outcome}`
- Hierarchy: **{r.status}**
- Research use only; not approved for clinical deployment.
- Stage 3B common-core ridge remains primary regardless of secondary performance.

## Intended use

Contemporaneous phenotype probability estimation at a full rheumatology assessment in patients resembling the development cohort. Not validated for autonomous diagnosis, prognosis, pre-examination screening, treatment decisions, or transport to another setting.

## Development data

- Patients: {int(r.n)}
- Events: {int(r.events)}
- Raw predictors: {int(r.raw_predictor_count)}
- Transformed features: {int(r.transformed_feature_count)}

## Final fitting

The complete preprocessing-plus-model pipeline was tuned by repeated stratified {CV_SPLITS}-fold CV × {CV_REPEATS} on all development patients using log loss, selected by the frozen one-standard-error stability rule, and refitted to all patients. Selected hyperparameters:

```json
{json.dumps(meta['selected_hyperparameters'],indent=2)}
```

No apparent full-data performance was calculated.

## Internal-validation evidence

Previously reported repeated nested OOF point estimates: AUROC {ev.auroc:.4f}, AP {ev.average_precision:.4f}, log loss {ev.log_loss:.4f}, Brier {ev.brier:.4f}. These development estimates do not establish external validity.

## Inputs and missingness

See `predictor_dictionary.csv`. Blank numeric/binary values are treated as unknown and imputed; they are not automatically recoded as absence. Categorical missingness uses `Unknown`. Explicit missingness indicators follow the frozen preprocessing specification.

## Limitations

Internal development only; no geographic, temporal, or prospective validation. Calibration and clinical utility may change in another workflow. No clinical threshold is supplied. For NoHand and Asymmetric models, contemporaneous joint/anatomy variables may be proximal to phenotype definition.

## Files

- `pipeline.joblib`
- `metadata.json`
- `predictor_dictionary.csv`
- `raw_predictors.csv`
- `transformed_features.csv`
- `tuning_candidates.csv`
- `example_input.csv`
- `expected_example_output.csv`
- `coefficients.csv` for ridge models
"""
 (mdir/'MODEL_CARD.md').write_text(card,encoding='utf-8')

In [ ]:
# 14. Portable batch prediction CLI
script=r"""from __future__ import annotations
import argparse,json,sys
from pathlib import Path
import joblib,numpy as np,pandas as pd
HERE=Path(__file__).resolve().parent
sys.path.insert(0,str(HERE))
import ra_final_model_components  # required for joblib deserialization

def predict(package,model_id,input_csv,output_csv):
    mdir=Path(package)/model_id;meta=json.loads((mdir/'metadata.json').read_text(encoding='utf-8'));raw=meta['raw_predictors'];x=pd.read_csv(input_csv)
    missing=[c for c in raw if c not in x.columns]
    if missing:raise ValueError(f'Missing required columns: {missing}')
    d=pd.read_csv(mdir/'predictor_dictionary.csv')
    for _,r in d.iterrows():
        c=r.predictor
        if r.semantic_type in ['numeric','binary']:
            converted=pd.to_numeric(x[c],errors='coerce');bad=x[c].notna()&converted.isna()
            if bad.any():raise ValueError(f'Non-numeric values in {c}')
            if np.isinf(converted.dropna()).any():raise ValueError(f'Infinite values in {c}')
            x[c]=converted
        elif r.allowed_categories_json!='[]':
            allowed=set(json.loads(r.allowed_categories_json));observed=set(x[c].dropna().astype(str));bad=observed-allowed
            if bad:raise ValueError(f'Unexpected categories in {c}: {sorted(bad)}')
    model=joblib.load(mdir/'pipeline.joblib');p=model.predict_proba(x[raw])[:,1]
    out=pd.DataFrame({'row_id':np.arange(len(x)),'probability':p});out.to_csv(output_csv,index=False);return out
if __name__=='__main__':
    a=argparse.ArgumentParser();a.add_argument('--package',default=str(HERE));a.add_argument('--model-id',required=True);a.add_argument('--input',required=True);a.add_argument('--output',required=True);z=a.parse_args();predict(z.package,z.model_id,z.input,z.output)
"""
(PACKAGE/'predict_stage4g.py').write_text(script,encoding='utf-8')

In [ ]:
# 15. Validate the portable CLI in fresh Python subprocesses
cli_rows=[]
for mid in model_registry.model_id:
 mdir=PACKAGE/mid;out=mdir/'cli_validation_output.csv'
 cmd=[sys.executable,str(PACKAGE/'predict_stage4g.py'),'--package',str(PACKAGE),'--model-id',mid,'--input',str(mdir/'example_input.csv'),'--output',str(out)]
 z=subprocess.run(cmd,text=True,capture_output=True)
 if z.returncode!=0:raise RuntimeError(f'CLI failed for {mid}: {z.stderr}')
 actual=pd.read_csv(out).probability.iloc[0];expected=pd.read_csv(mdir/'expected_example_output.csv').probability.iloc[0];diff=abs(actual-expected);cli_rows.append({'model_id':mid,'absolute_difference':diff,'cli_pass':diff<1e-8});out.unlink()
cli_validation=pd.DataFrame(cli_rows);cli_validation.to_csv(TABLES/'cli_prediction_validation.csv',index=False,encoding='utf-8-sig');display(cli_validation)

In [ ]:
# 16. External-validation specification
text="""# Frozen External-Validation Specification

1. Use the serialized pipeline unchanged on eligible external patients.
2. Confirm outcome definitions, predictor timing, units, categories, and missingness mechanisms before prediction.
3. Report missingness and exclusions before inspecting model performance.
4. First report original-model performance without recalibration or model updating.
5. Required measures: AUROC, average precision, log loss, Brier score, calibration-in-the-large, calibration slope, flexible calibration plot, and outcome-stratified probability distributions, all with patient-level confidence intervals.
6. Determine validation sample size for precision of calibration-in-the-large, calibration slope, AUROC, and any prespecified net benefit—not by a crude events-per-variable rule.
7. Do not select a clinical threshold unless a decision, intended user, setting, and relative false-positive/false-negative harms are defined in advance.
8. If updating is needed, preserve and report the untouched original-model evaluation first; then evaluate intercept-only, intercept-and-slope, or broader updating as separate analyses.
9. Assess performance across clinically relevant sites, calendar periods, sex, age, and missingness patterns when adequately sized.
10. Do not interpret predictive associations causally.
"""
(PACKAGE/'EXTERNAL_VALIDATION_SPECIFICATION.md').write_text(text,encoding='utf-8')

In [ ]:
# 17. Selection and package figures
selected=all_tuning[all_tuning.selected_by_one_se_rule].copy();selected['label']=selected.outcome+'\n'+selected.model
fig,ax=plt.subplots(figsize=(12,5));ax.scatter(range(len(selected)),selected.mean_cv_log_loss,color='#d95f0e',s=55);ax.vlines(range(len(selected)),selected.mean_cv_log_loss-selected.se_cv_log_loss,selected.mean_cv_log_loss+selected.se_cv_log_loss,color='#2c7fb8',lw=2);ax.set_xticks(range(len(selected)),selected.label,rotation=25,ha='right');ax.set_ylabel('Repeated-CV log loss ± SE');ax.set_title('Final full-cohort hyperparameter selection (not a performance estimate)');fig.tight_layout()
for e in ['png','svg']:fig.savefig(FIGURES/f'final_hyperparameter_selection.{e}',bbox_inches='tight',dpi=400 if e=='png' else None)
plt.show();plt.close(fig)

## Final package audit

In [ ]:
# 18. Automated Stage 4G audit
checks=[]
def check(i,d,c,x=''):checks.append({'check_id':i,'description':d,'status':'PASS' if bool(c) else 'FAIL','details':x})
check('G-001','Stage 3B audit passed',pd.read_csv(inputs['stage3b_audit']).status.eq('PASS').all())
check('G-002','Stage 3C audit passed',pd.read_csv(inputs['stage3c_audit']).status.eq('PASS').all())
check('G-003','Exactly eight frozen final models packaged',len(model_registry)==8,len(model_registry))
check('G-004','Three primary models are core ridge',len(model_registry.query("status=='primary' and model=='core_ridge'"))==3)
check('G-005','Three secondary models are broad ridge',len(model_registry.query("status=='secondary' and model=='refined_broad_ridge'"))==3)
check('G-006','Two exploratory challengers are XGBoost',len(model_registry.query("status=='exploratory_challenger' and model=='xgboost'"))==2)
check('G-007','Only permitted outcomes packaged',set(model_registry.outcome)==set(OUTCOMES))
check('G-008','OnlyLargeJoints excluded','OnlyLargeJoints' not in model_registry.outcome.unique())
check('G-009','Every final fit uses all 820 patients',model_registry.n.eq(820).all())
check('G-010','Natural prevalence and no resampling frozen',PROTOCOL['preprocessing']['class_weight'] is None and PROTOCOL['preprocessing']['resampling']=='none')
check('G-011','All selected candidates are within one SE',all_tuning.loc[all_tuning.selected_by_one_se_rule,'within_one_se'].all())
# ridge selection is the smallest eligible C
ridge_ok=True
for (o,m),g in all_tuning[all_tuning.model.isin(['core_ridge','refined_broad_ridge'])].groupby(['outcome','model']):
 eligible=g[g.within_one_se];cs=eligible.params_json.map(lambda s:json.loads(s)['model__C']);chosen=g[g.selected_by_one_se_rule].params_json.map(lambda s:json.loads(s)['model__C']).iloc[0];ridge_ok &= abs(chosen-cs.min())<1e-15
check('G-012','Ridge one-SE rule chooses strongest eligible regularization',ridge_ok)
check('G-013','All reload predictions reproduce exactly',reload_validation.reload_pass.all())
check('G-014','All CLI predictions reproduce exactly',cli_validation.cli_pass.all())
check('G-015','Every model has complete serialized pipeline',all((PACKAGE/m/'pipeline.joblib').exists() for m in model_registry.model_id))
check('G-016','Every model has metadata and model card',all((PACKAGE/m/'metadata.json').exists() and (PACKAGE/m/'MODEL_CARD.md').exists() for m in model_registry.model_id))
check('G-017','Every model has raw and transformed feature lists',all((PACKAGE/m/'raw_predictors.csv').exists() and (PACKAGE/m/'transformed_features.csv').exists() for m in model_registry.model_id))
check('G-018','Every model has predictor dictionary',all((PACKAGE/m/'predictor_dictionary.csv').exists() for m in model_registry.model_id))
check('G-019','Every model has synthetic input and expected output',all((PACKAGE/m/'example_input.csv').exists() and (PACKAGE/m/'expected_example_output.csv').exists() for m in model_registry.model_id))
check('G-020','Every ridge model has coefficients',all((PACKAGE/m/'coefficients.csv').exists() for m in model_registry.query("model!='xgboost'").model_id))
check('G-021','XGBoost limited to NoHand and Asymmetric',set(model_registry.query("model=='xgboost'").outcome)=={'NoHandJointInvolvement','AsymmetricArthritis'})
check('G-022','Raw models exclude outcome and VisitID',all(r.outcome not in json.loads((PACKAGE/r.model_id/'metadata.json').read_text())['raw_predictors'] and 'VisitID' not in json.loads((PACKAGE/r.model_id/'metadata.json').read_text())['raw_predictors'] for _,r in model_registry.iterrows()))
check('G-023','Portable component module distributed',(PACKAGE/'ra_final_model_components.py').exists())
check('G-024','Portable prediction CLI distributed',(PACKAGE/'predict_stage4g.py').exists())
check('G-025','External-validation specification distributed',(PACKAGE/'EXTERNAL_VALIDATION_SPECIFICATION.md').exists())
check('G-026','No apparent performance calculated',True,'Only prior OOF performance table is copied as evidence')
check('G-027','No clinical threshold supplied',True,'Probability output only')
check('G-028','Full/quick mode explicitly separated',OUT.name==('modeling_results_stage4g_quick' if QUICK else 'modeling_results_stage4g'))
check('G-029','Selection figure saved PNG and SVG',all((FIGURES/f'final_hyperparameter_selection.{e}').exists() for e in ['png','svg']))
check('G-030','Research-only status frozen',PROTOCOL['intended_use']['status'].startswith('Research-use-only'))
audit=pd.DataFrame(checks);audit.to_csv(TABLES/'stage4g_validation_audit.csv',index=False,encoding='utf-8-sig');display(audit)
if not audit.status.eq('PASS').all():raise AssertionError('Stage 4G audit failed')
print('ALL 30 STAGE 4G CHECKS PASSED')

In [ ]:
# 19. Create uniquely named audit-upload copies, then hash every deliverable
AUDIT_UPLOAD=OUT/'audit_upload';AUDIT_UPLOAD.mkdir(parents=True,exist_ok=True)
for mid in model_registry.model_id:
 mdir=PACKAGE/mid
 for source,suffix in [('pipeline.joblib','pipeline.joblib'),('metadata.json','metadata.json'),('MODEL_CARD.md','MODEL_CARD.md'),('tuning_candidates.csv','tuning_candidates.csv')]:shutil.copy2(mdir/source,AUDIT_UPLOAD/f'{mid}__{suffix}')
for source in ['final_model_registry.csv','reload_prediction_validation.csv','cli_prediction_validation.csv','prior_oof_performance_reference.csv','stage4g_validation_audit.csv']:
 shutil.copy2(TABLES/source,AUDIT_UPLOAD/source)
art=[]
for p in sorted([x for x in OUT.rglob('*') if x.is_file() and x.name not in ['artifact_registry.csv'] and 'report' not in x.parts]):art.append({'filename':p.name,'relative_path':str(p.relative_to(OUT)),'size_bytes':p.stat().st_size,'sha256':sha(p)})
artifacts=pd.DataFrame(art);artifacts.to_csv(TABLES/'artifact_registry.csv',index=False,encoding='utf-8-sig')
summary={'run_signature':RUN_SIGNATURE,'mode':'quick' if QUICK else 'full','models_packaged':8,'n_per_model':820,'cv_splits':CV_SPLITS,'cv_repeats':CV_REPEATS,'xgb_candidates':XGB_N,'all_30_audits_passed':True,'apparent_performance_reported':False,'clinical_use_allowed':False}
with open(OUT/'stage4g_run_summary.json','w') as f:json.dump(summary,f,indent=2)
print('Registered artifacts:',len(artifacts))

In [ ]:
# 20. FINAL CELL — checkpoint-backed self-contained HTML report
if os.environ.get('RA_STAGE4G_EXPORT_MODE')=='1':print('Export subprocess mode: recursive export skipped.')
else:
 candidates=[PROJECT_DIR/NOTEBOOK_NAME,ENV_PROJECT/NOTEBOOK_NAME,COLAB_PROJECT/NOTEBOOK_NAME,Path.cwd()/NOTEBOOK_NAME,Path.cwd()/'ra_analysis'/NOTEBOOK_NAME,Path.cwd()/NOTEBOOK_NAME];notebook=next((p.resolve() for p in candidates if p.exists()),None)
 if notebook is None:raise FileNotFoundError(f'Save notebook as {NOTEBOOK_NAME}')
 env=os.environ.copy();env['RA_STAGE4G_EXPORT_MODE']='1';env.pop('RA_STAGE4G_FORCE_RECOMPUTE',None);report=REPORT/'Stage4G_final_model_packaging_report.html'
 cmd=[sys.executable,'-m','jupyter','nbconvert','--to','html','--execute','--ExecutePreprocessor.timeout=0','--HTMLExporter.embed_images=True','--output',report.name,'--output-dir',str(REPORT),str(notebook)]
 z=subprocess.run(cmd,env=env,cwd=str(notebook.parent),text=True,capture_output=True)
 if z.stdout:print(z.stdout)
 if z.returncode:print(z.stderr);raise RuntimeError('HTML export failed')
 print('Report:',report.resolve())